In [1]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
CONFIG = {
    'xml_root': r'E:\DATA\Annotations',
    'video_root': r'E:\DATA\Videos', 
    
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'target_label': 'bangla-tesla', 
    'obs_len': 15,  
    'pred_len': 45, 
    
    # SGNet Specifics
    'hidden_size': 256,
    'embed_size': 64,
    'goal_weight': 1.0, # How much to focus on the final point accuracy
    
    'batch_size': 32,
    'epochs': 20,
    'lr': 1e-3
}

print(f"✅ SGNet Config Loaded. Device: {CONFIG['device']}")

✅ SGNet Config Loaded. Device: cpu


In [2]:
class IDDTrajectoryDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45, target_label='bangla-tesla'):
        self.obs_len = obs_len
        self.pred_len = pred_len
        self.seq_len = obs_len + pred_len
        self.samples = []
        
        print(f"📂 Parsing XMLs for label: '{target_label}'...")
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        
        for xml in tqdm(xml_files):
            try:
                tree = ET.parse(xml)
                root = tree.getroot()
                meta_size = root.find('meta').find('original_size')
                img_w = float(meta_size.find('width').text)
                img_h = float(meta_size.find('height').text)
                
                for track in root.findall('track'):
                    if track.attrib['label'] != target_label: continue
                    track_data = [] 
                    boxes = sorted(track.findall('box'), key=lambda b: int(b.attrib['frame']))
                    
                    for box in boxes:
                        if box.get('outside') == '1': continue
                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        w = xbr - xtl
                        h = ybr - ytl
                        cx = (xtl + xbr) / 2
                        cy = (ytl + ybr) / 2
                        track_data.append([cx/img_w, cy/img_h, w/img_w, h/img_h])
                    
                    track_data = np.array(track_data)
                    if len(track_data) < self.seq_len: continue
                    
                    stride = 10 
                    for i in range(0, len(track_data) - self.seq_len + 1, stride):
                        obs = track_data[i : i+obs_len]
                        pred = track_data[i+obs_len : i+obs_len+pred_len]
                        self.samples.append({
                            'obs': obs[:, 0:2], # [cx, cy]
                            'pred': pred        # [cx, cy, w, h]
                        })
            except: pass
            
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            torch.tensor(item['obs'], dtype=torch.float32), 
            torch.tensor(item['pred'], dtype=torch.float32)
        )

# Create Dataset
dataset = IDDTrajectoryDataset(CONFIG['xml_root'], obs_len=CONFIG['obs_len'], pred_len=CONFIG['pred_len'])
print(f"✅ Dataset Created: {len(dataset)} sequences.")

📂 Parsing XMLs for label: 'bangla-tesla'...


  0%|          | 0/6 [00:00<?, ?it/s]

✅ Dataset Created: 2156 sequences.


In [3]:
class SGNet(nn.Module):
    def __init__(self, input_size=2, output_size=4, hidden_size=256, embed_size=64):
        super(SGNet, self).__init__()
        
        # 1. Coordinate Embedding
        self.embed = nn.Linear(input_size, embed_size)
        self.relu = nn.ReLU()
        
        # 2. Encoder (Reads History)
        self.encoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        
        # 3. Goal Module (Predicts Final State at T=45)
        # Takes Encoder hidden state -> Outputs 1 step (x, y, w, h)
        self.goal_generator = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, output_size)
        )
        
        # 4. Decoder (Path Interpolator)
        # Takes (Embedding + Goal Info) as input
        self.decoder_cell = nn.LSTMCell(embed_size + output_size, hidden_size)
        self.out_fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, obs, pred_len):
        # obs: [Batch, 15, 2]
        batch_size = obs.size(0)
        
        # --- ENCODE HISTORY ---
        obs_emb = self.relu(self.embed(obs))
        _, (h_enc, c_enc) = self.encoder(obs_emb)
        
        # h_enc is [1, Batch, Hidden]. Squeeze to [Batch, Hidden]
        hidden_state = h_enc.squeeze(0)
        
        # --- PREDICT GOAL ---
        # "Where will it be at the very end?"
        pred_goal = self.goal_generator(hidden_state) # [Batch, 4] (x, y, w, h)
        
        # --- DECODE TRAJECTORY ---
        outputs = []
        
        # Initial Decoder State = Encoder State
        h_dec = h_enc.squeeze(0)
        c_dec = c_enc.squeeze(0)
        
        # First input to decoder is the last observed position
        curr_input = obs[:, -1, :].unsqueeze(1) # [Batch, 1, 2]
        
        for _ in range(pred_len):
            # Embed current input
            curr_emb = self.relu(self.embed(curr_input)).squeeze(1) # [Batch, Embed]
            
            # Key SGNet Feature: Concatenate Input with Predicted Goal
            # The decoder always knows where it needs to end up.
            dec_input = torch.cat([curr_emb, pred_goal], dim=1) # [Batch, Embed + 4]
            
            h_dec, c_dec = self.decoder_cell(dec_input, (h_dec, c_dec))
            
            pred_step = self.out_fc(h_dec) # [Batch, 4]
            outputs.append(pred_step.unsqueeze(1))
            
            # Next input
            curr_input = pred_step[:, :2].unsqueeze(1)
            
        pred_traj = torch.cat(outputs, dim=1)
        
        return pred_traj, pred_goal

# Initialize
model = SGNet(
    input_size=2,
    output_size=4,
    hidden_size=CONFIG['hidden_size'],
    embed_size=CONFIG['embed_size']
).to(CONFIG['device'])

print(f"✅ SGNet Model Initialized.")

✅ SGNet Model Initialized.


In [4]:
# Split Data
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False)

optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
criterion = nn.MSELoss()

print(f"🚀 Starting SGNet Training...")

for epoch in range(CONFIG['epochs']):
    model.train()
    train_loss = 0
    
    for obs, target in tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}"):
        obs = obs.to(CONFIG['device'])
        target = target.to(CONFIG['device'])
        
        optimizer.zero_grad()
        
        # Forward Pass
        pred_traj, pred_goal = model(obs, CONFIG['pred_len'])
        
        # Ground Truth Goal is the LAST frame of the target sequence
        gt_goal = target[:, -1, :] 
        
        # 1. Trajectory Loss (Full Sequence)
        loss_traj = criterion(pred_traj, target)
        
        # 2. Goal Loss (Final Endpoint)
        loss_goal = criterion(pred_goal, gt_goal)
        
        # Total Loss (Weighted sum)
        loss = loss_traj + (CONFIG['goal_weight'] * loss_goal)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for obs, target in val_loader:
            obs = obs.to(CONFIG['device'])
            target = target.to(CONFIG['device'])
            
            pred_traj, pred_goal = model(obs, CONFIG['pred_len'])
            gt_goal = target[:, -1, :]
            
            loss_t = criterion(pred_traj, target)
            loss_g = criterion(pred_goal, gt_goal)
            
            val_loss += (loss_t + CONFIG['goal_weight'] * loss_g).item()
            
    avg_train = train_loss / len(train_loader)
    avg_val = val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1} | Train Loss: {avg_train:.5f} | Val Loss: {avg_val:.5f}")

torch.save(model.state_dict(), "bangla_tesla_sgnet.pth")
print("💾 SGNet Model Saved.")

🚀 Starting SGNet Training...


Epoch 1:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.03652 | Val Loss: 0.01410


Epoch 2:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.01392 | Val Loss: 0.00857


...

Epoch 18 | Train Loss: 0.00658 | Val Loss: 0.00567


Epoch 19:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 19 | Train Loss: 0.00582 | Val Loss: 0.00430


Epoch 20:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 20 | Train Loss: 0.00571 | Val Loss: 0.00446
💾 SGNet Model Saved.


In [5]:
import pickle 

def calculate_sgnet_metrics(model, loader):
    model.eval()
    mse_traj_list, cmse_final_list, cfmse_final_list = [], [], []
    W_px, H_px = 2592, 1944 
    
    with torch.no_grad():
        for obs, target in loader:
            obs = obs.to(CONFIG['device'])
            target = target.cpu().numpy()
            
            # Predict
            pred_traj, _ = model(obs, CONFIG['pred_len'])
            preds = pred_traj.cpu().numpy()
            
            # --- Un-normalize ---
            pred_seq = np.zeros_like(preds)
            gt_seq = np.zeros_like(target)
            
            pred_seq[:, :, 0] = preds[:, :, 0] * W_px; pred_seq[:, :, 2] = preds[:, :, 2] * W_px
            pred_seq[:, :, 1] = preds[:, :, 1] * H_px; pred_seq[:, :, 3] = preds[:, :, 3] * H_px
            gt_seq[:, :, 0] = target[:, :, 0] * W_px; gt_seq[:, :, 2] = target[:, :, 2] * W_px
            gt_seq[:, :, 1] = target[:, :, 1] * H_px; gt_seq[:, :, 3] = target[:, :, 3] * H_px
            
            # --- Metrics ---
            # MSE
            traj_mse = np.mean(np.sum((pred_seq[:,:,:2] - gt_seq[:,:,:2])**2, axis=2), axis=1)
            mse_traj_list.extend(traj_mse)

            # C-MSE (Final Endpoint)
            c_mse = np.sum((pred_seq[:, -1, :2] - gt_seq[:, -1, :2])**2, axis=1)
            cmse_final_list.extend(c_mse)
            
            # CF-MSE
            gt_foot = np.stack([gt_seq[:, -1, 0], gt_seq[:, -1, 1] + gt_seq[:, -1, 3]/2], axis=1)
            pred_foot = np.stack([pred_seq[:, -1, 0], pred_seq[:, -1, 1] + pred_seq[:, -1, 3]/2], axis=1)
            foot_mse = np.sum((pred_foot - gt_foot)**2, axis=1)
            cfmse_final_list.extend(c_mse + foot_mse)

    return np.mean(mse_traj_list), np.mean(cmse_final_list), np.mean(cfmse_final_list)

print("📊 Calculating SGNet Metrics...")
mse, c_mse, cf_mse = calculate_sgnet_metrics(model, val_loader)

print(f"\n✅ SGNet Results (Bangla-Tesla):")
print(f"   MSE (Avg Trajectory): {mse:.2f}")
print(f"   C-MSE (Center @ 1.5s):  {c_mse:.2f}")
print(f"   CF-MSE (Center+Foot @ 1.5s): {cf_mse:.2f}")

with open('result_bangla_tesla_sgnet.pkl', 'wb') as f:
    pickle.dump({'MSE': mse, 'C-MSE': c_mse, 'CF-MSE': cf_mse}, f)

📊 Calculating SGNet Metrics...

✅ SGNet Results (Bangla-Tesla):
   MSE (Avg Trajectory): 13872.15
   C-MSE (Center @ 1.5s):  28258.27
   CF-MSE (Center+Foot @ 1.5s): 63870.88
